In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import mizatools

# Density over all kind of objects - emitters and not emitters

In [6]:
import pandas as pd
from pathlib import Path

outdir = '/mnt/hdcasa/splus_gaia/mc_catalogs/'


def concat_fields(outdir, pattern):

    pasta = Path(outdir)

    files = sorted(pasta.glob(pattern))

    full = pd.concat(
        (pd.read_csv(f) for f in files),
        ignore_index=True
    )
    return full

In [7]:
mc_fields = concat_fields(outdir, 'MC*.csv')

/tmp/ipykernel_5487/2480420889.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full = pd.concat(


In [8]:
mc_fields.size

1210689747

In [9]:
mc_fields_clean = mc_fields[(mc_fields['err_mag_psf_j0660'] < 0.2)&
        (mc_fields['err_mag_psf_i'] < 0.2)&
        (mc_fields['err_mag_psf_r'] < 0.2)&
        (mc_fields['mag_psf_j0660']/mc_fields['err_mag_psf_j0660'] > 10)&
        (mc_fields['mag_psf_i']/mc_fields['err_mag_psf_i'] > 10)&
        (mc_fields['mag_psf_r']/mc_fields['err_mag_psf_r'] > 10)&
        mc_fields['mag_psf_r'].between(13,19.5)]

In [10]:
mc_emitters_all = pd.read_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/halpha_emitters_mc_simbad_and_all_dist.csv')

In [11]:
mc_fields_clean["h_alpha_emitters_8sigma"] = mc_fields_clean["id"].isin(
    mc_emitters_all["splus_idr6_id"].unique()
)

/tmp/ipykernel_5487/3485438129.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mc_fields_clean["h_alpha_emitters_8sigma"] = mc_fields_clean["id"].isin(


In [12]:
mc_fields_clean['s2noise_r'] = mc_fields_clean['mag_psf_r']/mc_fields_clean['err_mag_psf_r']

/tmp/ipykernel_5487/4066187811.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mc_fields_clean['s2noise_r'] = mc_fields_clean['mag_psf_r']/mc_fields_clean['err_mag_psf_r']


In [13]:
mc_fields_clean = mc_fields_clean.sort_values('s2noise_r', ascending=False)

In [14]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

keep_mask = mizatools.unique_by_sep(
    mc_fields_clean["ra"].values,
    mc_fields_clean["dec"].values,
    min_sep_arcsec=2.0
)

mc_fields_clean["unique_values_kept"] = keep_mask.astype(int)

In [27]:
mc_fields_clean.to_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/all_detections_mc.csv', index=False)

## selecting MC field range

In [ ]:
fields = pd.read_csv('/home/shared/splus_gaia/data/field_catalogs/dr6_list.csv')

fields = fields[fields['field'].str.contains('MC')]

In [ ]:
fields['ra'].describe()

In [ ]:
fields['dec'].describe()

## Aperture Method

In [ ]:
import numpy as np

def sample_objects_in_nonoverlapping_apertures(
    df,
    ra_col: str,
    dec_col: str,
    ra_min: float,
    ra_max: float,
    dec_min: float,
    dec_max: float,
    n_apertures: int = 100,
    radius_arcmin: float = 1.0,
    cols: list[str] | None = None,
    seed: int | None = 0,
    max_tries: int = 200000,
    return_type: str = "pandas",  # "pandas" ou "polars"
):
    """
    Cria N aberturas circulares aleatórias de mesmo raio, NÃO sobrepostas,
    dentro de uma caixa (ra_min/ra_max/dec_min/dec_max), e retorna os objetos
    dentro de cada abertura.

    Não sobreposição: distância entre centros >= 2*raio.

    Retorna:
      apertures: list[dict] com centros/raio
      results:   list[DF] (1 DF por abertura) com objetos dentro, + metadados
    """
    from astropy.coordinates import SkyCoord
    import astropy.units as u

    # --- detectar pandas vs polars e pegar arrays
    is_polars = df.__class__.__module__.startswith("polars")
    if is_polars:
        import polars as pl
        ra = df[ra_col].to_numpy()
        dec = df[dec_col].to_numpy()
        if cols is None:
            cols = df.columns
        if ra_col not in cols: cols = [ra_col] + cols
        if dec_col not in cols: cols = [dec_col] + cols
    else:
        ra = df[ra_col].to_numpy()
        dec = df[dec_col].to_numpy()
        if cols is None:
            cols = list(df.columns)
        if ra_col not in cols: cols = [ra_col] + cols
        if dec_col not in cols: cols = [dec_col] + cols

    ra = np.asarray(ra, float)
    dec = np.asarray(dec, float)
    cat = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")

    r = (radius_arcmin * u.arcmin)
    r_deg = r.to(u.deg).value

    # margem para garantir círculo inteiro dentro da caixa
    dec_abs_max = max(abs(dec_min), abs(dec_max))
    cos_min = np.cos(np.deg2rad(dec_abs_max))
    cos_min = max(cos_min, 1e-6)
    ra_margin = r_deg / cos_min

    ra_lo = ra_min + ra_margin
    ra_hi = ra_max - ra_margin
    dec_lo = dec_min + r_deg
    dec_hi = dec_max - r_deg

    if ra_lo >= ra_hi or dec_lo >= dec_hi:
        raise ValueError(
            "A área é pequena demais para acomodar a abertura com margem. "
            "Diminua radius_arcmin ou aumente a área."
        )

    rng = np.random.default_rng(seed)

    # separação mínima entre centros para não sobrepor círculos
    min_center_sep = (2.0 * r)  # 2*raio

    centers_list = []  # SkyCoord individuais
    apertures = []
    results = []

    # --- loop de amostragem por rejeição
    tries = 0
    while len(centers_list) < n_apertures and tries < max_tries:
        tries += 1

        cra = rng.uniform(ra_lo, ra_hi)
        cdec = rng.uniform(dec_lo, dec_hi)
        c0 = SkyCoord(ra=cra*u.deg, dec=cdec*u.deg, frame="icrs")

        if centers_list:
            prev = SkyCoord(centers_list)
            # checa se está perto demais de algum centro já aceito
            if np.any(c0.separation(prev) < min_center_sep):
                continue

        centers_list.append(c0)

    if len(centers_list) < n_apertures:
        raise RuntimeError(
            f"Não consegui posicionar {n_apertures} aberturas não sobrepostas "
            f"nessa área com raio={radius_arcmin} arcmin. "
            f"Consegui {len(centers_list)} após {tries} tentativas. "
            "Sugestões: aumente a área, reduza o raio, ou reduza n_apertures."
        )

    # --- agora seleciona objetos dentro de cada abertura
    for k, c0 in enumerate(centers_list):
        sep = cat.separation(c0)
        m = sep <= r

        apertures.append({
            "aperture_id": k,
            "center_ra": float(c0.ra.deg),
            "center_dec": float(c0.dec.deg),
            "radius_arcmin": float(radius_arcmin),
        })

        if is_polars:
            import polars as pl
            sub = df.select(cols).filter(pl.Series(m))
            sub = sub.with_columns([
                pl.lit(k).alias("aperture_id"),
                pl.lit(float(c0.ra.deg)).alias("center_ra"),
                pl.lit(float(c0.dec.deg)).alias("center_dec"),
                pl.Series(sep.arcsec[m]).alias("sep_arcsec"),
            ])
            if return_type == "pandas":
                sub = sub.to_pandas()
        else:
            sub = df.loc[m, cols].copy()
            sub["aperture_id"] = k
            sub["center_ra"] = float(c0.ra.deg)
            sub["center_dec"] = float(c0.dec.deg)
            sub["sep_arcsec"] = sep.arcsec[m]

            if return_type == "polars":
                import polars as pl
                sub = pl.from_pandas(sub)

        results.append(sub)

    return apertures, results

In [ ]:
cols = ['id', 'ra', 'dec', 'mag_psf_g', 'mag_psf_i', 'mag_psf_j0378',
       'mag_psf_j0395', 'mag_psf_j0410', 'mag_psf_j0430', 'mag_psf_j0515',
       'mag_psf_j0660', 'mag_psf_j0861', 'mag_psf_r', 'mag_psf_u', 'mag_psf_z',
       'err_mag_psf_g', 'err_mag_psf_i', 'err_mag_psf_j0378',
       'err_mag_psf_j0395', 'err_mag_psf_j0410', 'err_mag_psf_j0430',
       'err_mag_psf_j0515', 'err_mag_psf_j0660', 'err_mag_psf_j0861',
       'err_mag_psf_r', 'err_mag_psf_u', 'err_mag_psf_z']

apertures, results = sample_objects_in_nonoverlapping_apertures(
    df=mc_fields_clean,
    ra_col="ra",
    dec_col="dec",
    ra_min=3.954166667, ra_max=96.49583333,
    dec_min=-74.6925, dec_max=-63.26777778,
    n_apertures=1000,
    radius_arcmin=1.0,
    cols=cols,
    seed=42,
    return_type="pandas",
)

import pandas as pd
all_in_apertures = pd.concat(results, ignore_index=True)

In [ ]:
plt.scatter(all_in_apertures['center_ra'], all_in_apertures['center_dec'])

## Matching detections to clusters

In [16]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord, search_around_sky, Angle
import astropy.units as u

def match_emitters_to_clusters_chunked(
    clusters: pd.DataFrame,
    stars: pd.DataFrame,
    *,
    cluster_ra_col: str = "ra",
    cluster_dec_col: str = "dec",
    cluster_amaj_col: str = "amaj",
    stars_ra_col: str = "ra",
    stars_dec_col: str = "dec",
    amaj_unit: str = "arcmin",
    keep_cols_clusters: list | None = None,
    keep_cols_stars: list | None = None,
    stars_chunk_size: int = 250_000,
    return_columns: str = "full",  # "full" | "pairs" (só indices + separação)
    sort_by_sep: bool = True,
) -> pd.DataFrame:
    """
    Versão otimizada em memória:
    - clusters ficam fixos em memória
    - estrelas são processadas em chunks (evita SkyCoord gigante)
    - acumula resultados por chunk e concatena no final

    return_columns:
      - "pairs": retorna apenas cluster_index, star_index, sep_arcsec, sep_arcmin
      - "full" : adiciona colunas prefixadas cluster_/star_ como na sua função
    """
    if keep_cols_clusters is None:
        keep_cols_clusters = []
    if keep_cols_stars is None:
        keep_cols_stars = []

    # --------
    # 1) Pré-filtra clusters (sem cópias pesadas)
    # --------
    cl_req = [cluster_ra_col, cluster_dec_col, cluster_amaj_col] + keep_cols_clusters
    cl_df = clusters.loc[:, cl_req]

    # máscara booleana (bem mais leve que replace/dropna em grandes tabelas)
    cl_mask = (
        np.isfinite(cl_df[cluster_ra_col].to_numpy(dtype=float, copy=False)) &
        np.isfinite(cl_df[cluster_dec_col].to_numpy(dtype=float, copy=False)) &
        np.isfinite(cl_df[cluster_amaj_col].to_numpy(dtype=float, copy=False)) &
        (cl_df[cluster_amaj_col].to_numpy(dtype=float, copy=False) > 0)
    )
    cl_df = cl_df.loc[cl_mask]

    if cl_df.empty:
        return pd.DataFrame(columns=["cluster_index","star_index","sep_arcsec","sep_arcmin"])

    # SkyCoord dos clusters (normalmente pequeno)
    cl_ra = cl_df[cluster_ra_col].to_numpy(dtype=np.float64, copy=False)
    cl_dec = cl_df[cluster_dec_col].to_numpy(dtype=np.float64, copy=False)
    c_clusters = SkyCoord(ra=cl_ra * u.deg, dec=cl_dec * u.deg)

    unit = u.Unit(amaj_unit)
    cl_amaj = cl_df[cluster_amaj_col].to_numpy(dtype=np.float64, copy=False)
    max_amaj = float(np.nanmax(cl_amaj))
    seplimit = Angle(max_amaj, unit)

    # Para mapear índices internos do cl_df (0..n-1) para índices originais do clusters
    cl_index_orig = cl_df.index.to_numpy()

    # --------
    # 2) Itera estrelas em chunks
    # --------
    st_req = [stars_ra_col, stars_dec_col] + keep_cols_stars
    st_df_all = stars.loc[:, st_req]

    out_chunks = []
    n = len(st_df_all)

    for start in range(0, n, stars_chunk_size):
        end = min(start + stars_chunk_size, n)
        st_chunk = st_df_all.iloc[start:end]

        # filtra chunk (sem replace/dropna)
        st_ra = st_chunk[stars_ra_col].to_numpy(dtype=np.float64, copy=False)
        st_dec = st_chunk[stars_dec_col].to_numpy(dtype=np.float64, copy=False)
        st_mask = np.isfinite(st_ra) & np.isfinite(st_dec)

        if not np.any(st_mask):
            continue

        st_chunk_valid = st_chunk.loc[st_mask]
        st_ra_v = st_chunk_valid[stars_ra_col].to_numpy(dtype=np.float64, copy=False)
        st_dec_v = st_chunk_valid[stars_dec_col].to_numpy(dtype=np.float64, copy=False)

        c_stars = SkyCoord(ra=st_ra_v * u.deg, dec=st_dec_v * u.deg)

        # busca por chunk
        idx_cl, idx_st, sep2d, _ = search_around_sky(c_clusters, c_stars, seplimit=seplimit)
        if len(idx_cl) == 0:
            continue

        # corte par-a-par com amaj do cluster
        amaj_per_pair = Angle(cl_amaj[idx_cl], unit)
        keep = sep2d <= amaj_per_pair
        if not np.any(keep):
            continue

        idx_cl_k = idx_cl[keep]
        idx_st_k = idx_st[keep]
        sep_k = sep2d[keep]

        # índices originais (clusters e stars)
        pairs = pd.DataFrame({
            "cluster_index": cl_index_orig[idx_cl_k],
            "star_index": st_chunk_valid.index.to_numpy()[idx_st_k],
            "sep_arcsec": sep_k.arcsec,
            "sep_arcmin": sep_k.to(u.arcmin).value,
        })

        if return_columns == "pairs":
            out_chunks.append(pairs)
            continue

        # "full": anexa metadados (cuidado: isso pode ficar grande se houver muitos pares)
        cl_out = clusters.loc[pairs["cluster_index"], [cluster_ra_col, cluster_dec_col, cluster_amaj_col] + keep_cols_clusters]
        cl_out = cl_out.add_prefix("cluster_").reset_index(drop=True)

        st_out = stars.loc[pairs["star_index"], [stars_ra_col, stars_dec_col] + keep_cols_stars]
        st_out = st_out.add_prefix("star_").reset_index(drop=True)

        out_chunks.append(pd.concat([pairs.reset_index(drop=True), cl_out, st_out], axis=1))

    if not out_chunks:
        return pd.DataFrame(columns=["cluster_index","star_index","sep_arcsec","sep_arcmin"])

    result = pd.concat(out_chunks, ignore_index=True)

    if sort_by_sep and "sep_arcmin" in result.columns and len(result) > 1:
        result = result.sort_values("sep_arcmin", ascending=True, kind="mergesort").reset_index(drop=True)

    return result

In [17]:
bica =pd.read_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/FullCatalogBica_08_20_Vizier.csv')

#bica_c = bica[bica['Type'].isin(['C', 'CN', 'CA', 'CC', 'NC' ])]  # apenas aglomerados do tipo "C"

In [ ]:
bica['[M/H]'].notna().sum()

In [ ]:
bica.columns

In [18]:
# clusters_df: colunas ["name","ra","dec","amaj"] (amaj em arcmin, por ex.)
# emitters_df: colunas ["source_id","ra","dec", ...]
out = match_emitters_to_clusters_chunked(
    bica,
    mc_fields_clean[mc_fields_clean["unique_values_kept"] == 1],
    cluster_ra_col="ra",
    cluster_dec_col="dec",
    cluster_amaj_col="amaj",
    stars_ra_col="ra",
    stars_dec_col="dec",
    amaj_unit="arcmin",
    keep_cols_clusters=["Names", "Type", "[M/H]", 'r_[M/H]'],         # traga o nome do aglomerado
    keep_cols_stars=["id"]  # traga id/atributos da estrela
)

In [19]:
mc_fields_clean["outside_cluster"] = ~mc_fields_clean["id"].isin(
    out["star_id"].unique()
)

In [25]:
len(out)

1639296

In [20]:
mc_fields_clean.columns

Index(['id', 'ra', 'dec', 'mag_psf_g', 'mag_psf_i', 'mag_psf_j0378',
       'mag_psf_j0395', 'mag_psf_j0410', 'mag_psf_j0430', 'mag_psf_j0515',
       'mag_psf_j0660', 'mag_psf_j0861', 'mag_psf_r', 'mag_psf_u', 'mag_psf_z',
       'err_mag_psf_g', 'err_mag_psf_i', 'err_mag_psf_j0378',
       'err_mag_psf_j0395', 'err_mag_psf_j0410', 'err_mag_psf_j0430',
       'err_mag_psf_j0515', 'err_mag_psf_j0660', 'err_mag_psf_j0861',
       'err_mag_psf_r', 'err_mag_psf_u', 'err_mag_psf_z',
       'h_alpha_emitters_8sigma', 's2noise_r', 'unique_values_kept',
       'outside_cluster'],
      dtype='object')

In [21]:
out.columns

Index(['cluster_index', 'star_index', 'sep_arcsec', 'sep_arcmin', 'cluster_ra',
       'cluster_dec', 'cluster_amaj', 'cluster_Names', 'cluster_Type',
       'cluster_[M/H]', 'cluster_r_[M/H]', 'star_ra', 'star_dec', 'star_id'],
      dtype='object')

In [22]:
mc_fields_clean = mc_fields_clean.merge(
    out,
    left_on="id",
    right_on="star_id",
    how="left"
)

In [26]:
len(mc_fields_clean)

9851341

In [28]:
bica

,Names,PA,bmin,l_logAge,logAge,r_logAge,[M/H],r_[M/H],PapI,B18,ra,dec,amaj,SimbadName,Type
0,BMS258,NaN,1.00,NaN,6.00,BGB+18,NaN,NaN,0.0,B18,16.795833,-72.586667,1.00,[BGB2018]SMC-M2-258,AC
1,BMS401,NaN,0.79,NaN,6.34,BGB+18,NaN,NaN,0.0,B18,17.490833,-72.157222,0.79,[BGB2018]SMC-M2-401,A
2,BMS376,NaN,1.32,NaN,6.40,BGB+18,NaN,NaN,0.0,B18,15.787083,-72.235556,1.32,[BGB2018]SMC-M2-376,A
3,IC1644,NaN,0.65,NaN,6.44,SBC+95,NaN,NaN,1.0,B18,17.305000,-73.193611,0.80,IC1644,NC
4,SSN8,NaN,0.20,NaN,6.48,SSN+07,NaN,NaN,2.0,B18,14.787500,-72.183333,0.20,[SSN2007]Sc8,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5869,ZHT-SP5,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78.383333,-67.174444,0.60,NaN,CN
5870,ZHT12,160.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79.175000,-67.803056,0.65,ZHTAN12,C
5871,ZHT2,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.620833,-66.610833,0.60,ZHTAN2,CA
5872,ZHT3,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.475000,-66.733333,1.50,ZHTAN3,C


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


inside = mc_fields_clean[(mc_fields_clean['outside_cluster'] == 0) & (mc_fields_clean["unique_values_kept"] == 1)]
outside = mc_fields_clean[(mc_fields_clean['outside_cluster'] == 1) & (mc_fields_clean["unique_values_kept"] == 1)]

# Criar figura com 3 subplots na vertical
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 18))

# Dentro dos clusters
h1 = ax1.hist2d(inside['ra'], inside['dec'], bins=500, 
                cmap='turbo', vmin=0)
ax1.set_title(f'Dentro de clusters ({len(inside)} objetos)', fontsize=14, fontweight='bold')
ax1.set_xlabel('RA (graus)', fontsize=12)
ax1.set_ylabel('Dec (graus)', fontsize=12)
ax1.tick_params(labelsize=10)
plt.colorbar(h1[3], ax=ax1, label='Densidade')

# Fora dos clusters
h2 = ax2.hist2d(outside['ra'], outside['dec'], bins=500, 
                cmap='magma', vmin=0)
ax2.set_title(f'Fora de clusters ({len(outside)} objetos)', fontsize=14, fontweight='bold')
ax2.set_xlabel('RA (graus)', fontsize=12)
ax2.set_ylabel('Dec (graus)', fontsize=12)
ax2.tick_params(labelsize=10)
plt.colorbar(h2[3], ax=ax2, label='Densidade')

# Todos juntos
h3 = ax3.hist2d(mc_fields_clean[mc_fields_clean["unique_values_kept"] == 1]['ra'], mc_fields_clean[mc_fields_clean["unique_values_kept"] == 1]['dec'], bins=500, 
                cmap='viridis')
ax3.set_title(f'Todos os objetos ({len(mc_fields_clean[mc_fields_clean["unique_values_kept"] == 1])} objetos)', fontsize=14, fontweight='bold')
ax3.set_xlabel('RA (graus)', fontsize=12)
ax3.set_ylabel('Dec (graus)', fontsize=12)
ax3.tick_params(labelsize=10)
plt.colorbar(h3[3], ax=ax3, label='Densidade')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mc_used = mc_fields_clean[mc_fields_clean["unique_values_kept"] == 1]

if 'outside_cluster' in mc_used.columns and 'h_alpha_emitters_8sigma' in mc_used.columns:
    
    # Separar todos os grupos
    inside = mc_used[mc_used['outside_cluster'] == 0]
    outside = mc_used[mc_used['outside_cluster'] == 1]
    
    # Grupos combinados
    inside_emitters = inside[inside['h_alpha_emitters_8sigma'] == 1]
    inside_non_emitters = inside[inside['h_alpha_emitters_8sigma'] == 0]
    outside_emitters = outside[outside['h_alpha_emitters_8sigma'] == 1]
    outside_non_emitters = outside[outside['h_alpha_emitters_8sigma'] == 0]
    
    # Criar figura com 7 gráficos
    fig, axes = plt.subplots(7, 1, figsize=(20, 50))
    
    # Configurações para cada gráfico
    plots_config = [
        {'data': inside, 'title': f'Dentro dos aglomerados estelares ({len(inside)})', 'cmap': 'viridis'},
        {'data': outside, 'title': f'Fora dos aglomerados estelares ({len(outside)})', 'cmap': 'viridis'},
        {'data': mc_used, 'title': f'Todos objetos ({len(mc_used)})', 'cmap': 'viridis'},
        {'data': inside_emitters, 'title': f'Emissores Hα dentro dos aglomerados estelares ({len(inside_emitters)})', 'cmap': 'viridis'},
        {'data': inside_non_emitters, 'title': f'Não emissores Hα dentro dos aglomerados estelares ({len(inside_non_emitters)})', 'cmap': 'viridis'},
        {'data': outside_emitters, 'title': f'Emissores Hα fora dos aglomerados estelares ({len(outside_emitters)})', 'cmap': 'viridis'},
        {'data': outside_non_emitters, 'title': f'Não emissores Hα fora dos aglomerados estelares ({len(outside_non_emitters)})', 'cmap': 'viridis'}
    ]
    
    # Criar cada gráfico
    for i, (ax, config) in enumerate(zip(axes, plots_config)):
        if len(config['data']) > 0:
            h = ax.hist2d(config['data']['ra'], config['data']['dec'], 
                         bins=500, cmap=config['cmap'], vmin=0)
            ax.set_title(config['title'], fontsize=13, fontweight='bold')
            cbar = plt.colorbar(h[3], ax=ax)
            cbar.set_label('Counts', fontsize=10)
        else:
            ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center', 
                   fontsize=12, transform=ax.transAxes)
            ax.set_title(config['title'] + ' (vazio)', fontsize=13, fontweight='bold')
        
        ax.set_xlabel('RA (graus)', fontsize=11)
        ax.set_ylabel('Dec (graus)', fontsize=11)
        ax.tick_params(labelsize=9)
        ax.grid(alpha=0.2)
    
    plt.suptitle('Análise Completa: Clusters + Emissores Hα', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.35)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

mc_used = mc_fields_clean[(mc_fields_clean["unique_values_kept"] == 1) & (mc_fields_clean['h_alpha_emitters_8sigma'] == 1)]

# Configuração HEALPix
nside = 512
npix = hp.nside2npix(nside)
pixel_area_deg2 = hp.nside2pixarea(nside, degrees=True)

def create_density_map(data, ra_col="ra", dec_col="dec"):
    if len(data) == 0:
        return np.zeros(npix, dtype=float)

    theta = np.deg2rad(90.0 - data[dec_col].to_numpy())  # colatitude
    phi   = np.deg2rad(data[ra_col].to_numpy() % 360.0)  # longitude
    pix = hp.ang2pix(nside, theta, phi)
    counts = np.bincount(pix, minlength=npix).astype(float)
    return counts / pixel_area_deg2

def healpix_density_stats(density_map):
    """
    Estatísticas do mapa de densidade (objetos/deg²).
    Considera apenas pixels com densidade > 0 como 'área coberta'.
    """
    m = np.asarray(density_map, float)
    nonzero = m > 0
    area_deg2 = nonzero.sum() * pixel_area_deg2

    stats = {
        "npix_total": int(m.size),
        "npix_nonzero": int(nonzero.sum()),
        "area_covered_deg2": float(area_deg2),
        "mean_density_nonzero": float(m[nonzero].mean()) if nonzero.any() else 0.0,
        "median_density_nonzero": float(np.median(m[nonzero])) if nonzero.any() else 0.0,
        "p90_density_nonzero": float(np.percentile(m[nonzero], 90)) if nonzero.any() else 0.0,
        "p99_density_nonzero": float(np.percentile(m[nonzero], 99)) if nonzero.any() else 0.0,
        "max_density": float(m.max()) if m.size else 0.0,
        "total_objects_est": float(m.sum() * pixel_area_deg2),  # integra densidade
    }
    return stats

# Separar dados
inside  = mc_used[mc_used["outside_cluster"] == 0]
outside = mc_used[mc_used["outside_cluster"] == 1]

density_in  = create_density_map(inside)
density_out = create_density_map(outside)

# Estatísticas
stats_in = healpix_density_stats(density_in)
stats_out = healpix_density_stats(density_out)

# Centro das MC
rot = (50, -69, 45)

# Zoom
xsize = 500
reso  = 5.0  # arcmin/pixel

plt.figure(figsize=(14, 6))

hp.gnomview(
    density_in,
    rot=rot,
    xsize=xsize,
    reso=reso,
    title=f'Dentro dos Aglomerados\n{len(inside):,} objetos',
    unit='objetos/deg²',
    cmap='viridis',
    min=0,
    sub=(1, 2, 1),
    notext=False,   # ✅ deixa o healpy desenhar ticks/labels de coord
    cbar=True
)

hp.gnomview(
    density_out,
    rot=rot,
    xsize=xsize,
    reso=reso,
    title=f'Fora dos Aglomerados\n{len(outside):,} objetos',
    unit='objetos/deg²',
    cmap='plasma',
    min=0,
    sub=(1, 2, 2),
    notext=False,
    cbar=True
)

# ✅ Pega os eixos corretos criados pelo healpy e adiciona grid
fig = plt.gcf()
axes = fig.get_axes()

# Os dois primeiros eixos costumam ser os mapas; as colorbars virão depois.
# Vamos tentar pegar os dois primeiros "Axes" que não são colorbar.
map_axes = [ax for ax in axes if ax.get_label() != '<colorbar>'][:2]

for ax in map_axes:
    ax.grid(alpha=0.3)

plt.suptitle(f'Mapas HEALPix NSIDE={nside} - Nuvens de Magalhães (zoom)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- imprime estatísticas
def pretty_print_stats(name, s):
    print(f"\n=== {name} ===")
    print(f"npix_nonzero        : {s['npix_nonzero']:,} / {s['npix_total']:,}")
    print(f"area_covered_deg2   : {s['area_covered_deg2']:.3f}")
    print(f"total_objects_est   : {s['total_objects_est']:.0f}")
    print(f"mean_density        : {s['mean_density_nonzero']:.3f} obj/deg²")
    print(f"median_density      : {s['median_density_nonzero']:.3f} obj/deg²")
    print(f"p90_density         : {s['p90_density_nonzero']:.3f} obj/deg²")
    print(f"p99_density         : {s['p99_density_nonzero']:.3f} obj/deg²")
    print(f"max_density         : {s['max_density']:.3f} obj/deg²")

pretty_print_stats("DENTRO aglomerados", stats_in)
pretty_print_stats("FORA aglomerados", stats_out)

In [ ]:
td = pd.read_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/Be_stars/splus_figueiredo_objects_all.csv')

bes = pd.read_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/Be_stars/be_stars_smc_lmc.csv')

bes['OGLE'] = bes['OGLE'].str.upper()

be = pd.merge(td, bes, right_on='OGLE', left_on='OGLE-II', how='inner')

In [ ]:
mc_fields_clean.columns

In [ ]:
mc_fields_clean.to_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/mc_all_additional_info.csv', index=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Configurar a figura
plt.figure(figsize=(10, 6))
mc_fields_outside = mc_fields_clean[(mc_fields_clean['outside_cluster'] == True) & (mc_fields_clean["unique_values_kept"] == 1)& (mc_fields_clean['h_alpha_emitters_8sigma'] == 1)]
# Plotar o scatter dos dados principais
sns.scatterplot(y=mc_fields_outside['mag_psf_j0515'] - mc_fields_outside['mag_psf_j0660'], 
                x=mc_fields_outside['mag_psf_j0660'] - mc_fields_outside['mag_psf_j0861'], 
                label='mc_fields_clean Hα Emitters', color='gray', s=20, alpha=0.6)
# Plotar o scatter das estrelas Be
sns.scatterplot(y=be['mag_psf_j0515'] - be['mag_psf_j0660'], 
                x=be['mag_psf_j0660'] - be['mag_psf_j0861'], 
                label='Estrelas Be - Figueiredo et. al 2025', color='blue', s=60, 
                marker='*', edgecolor='black')

# Adicionar o contorno (densidade) para os dados 'be'
# Opção 1: Contorno de densidade (recomendado)
sns.kdeplot(x=be['mag_psf_j0660'] - be['mag_psf_j0861'],
            y=be['mag_psf_j0515'] - be['mag_psf_j0660'],
            color='blue', linewidths=3, levels=5)

# Opção 2: Se preferir contornos mais suaves
# sns.kdeplot(x=be['mag_psf_j0660'] - be['mag_psf_j0861'],
#             y=be['mag_psf_j0515'] - be['mag_psf_j0660'],
#             fill=True, alpha=0.3, color='blue', thresh=0.1)

# Opção 3: Se quiser um contorno com hexbin (para muitos pontos)
# plt.hexbin(x=be['mag_psf_j0660'] - be['mag_psf_j0861'],
#            y=be['mag_psf_j0515'] - be['mag_psf_j0660'],
#            gridsize=30, cmap='Blues', alpha=0.7)

# Configurações dos eixos
plt.ylim(5, -5)
plt.xlim(3, -5)

plt.axvline(-0.15)
plt.axhline(0.3)

# Rótulos e título
plt.xlabel('F0660 - F0861', fontsize=12)
plt.ylabel('F0515 - F0660', fontsize=12)
plt.title('Diagrama de Cores - Hα Emitters e Estrelas Be (Figueiredo et. al, 2025 & Simbad)', fontsize=14)

# Adicionar grade para melhor visualização
plt.grid(True, alpha=0.3)

# Legenda
plt.legend(loc='best', fontsize=10)

# Ajustar layout
plt.tight_layout()

plt.show()

In [ ]:
be_selection_field = mc_fields_clean[(mc_fields_clean['outside_cluster'] == 1) &
                                      (mc_fields_clean["unique_values_kept"] == 1)&
                                        (mc_fields_clean['h_alpha_emitters_8sigma'] == 1) &
                                           (mc_fields_clean['mag_psf_j0515'] - mc_fields_clean['mag_psf_j0660'] < 0.3) &
                                            (mc_fields_clean['mag_psf_j0660'] - mc_fields_clean['mag_psf_j0861'] < -0.15)]

In [ ]:
be_selection_field

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

mc_used = be_selection_field

# Configuração HEALPix
nside = 512
npix = hp.nside2npix(nside)
pixel_area_deg2 = hp.nside2pixarea(nside, degrees=True)

def create_density_map(data, ra_col="ra", dec_col="dec"):
    if len(data) == 0:
        return np.zeros(npix, dtype=float)

    theta = np.deg2rad(90.0 - data[dec_col].to_numpy())  # colatitude
    phi   = np.deg2rad(data[ra_col].to_numpy() % 360.0)  # longitude
    pix = hp.ang2pix(nside, theta, phi)
    counts = np.bincount(pix, minlength=npix).astype(float)
    return counts / pixel_area_deg2

def healpix_density_stats(density_map):
    """
    Estatísticas do mapa de densidade (objetos/deg²).
    Considera apenas pixels com densidade > 0 como 'área coberta'.
    """
    m = np.asarray(density_map, float)
    nonzero = m > 0
    area_deg2 = nonzero.sum() * pixel_area_deg2

    stats = {
        "npix_total": int(m.size),
        "npix_nonzero": int(nonzero.sum()),
        "area_covered_deg2": float(area_deg2),
        "mean_density_nonzero": float(m[nonzero].mean()) if nonzero.any() else 0.0,
        "median_density_nonzero": float(np.median(m[nonzero])) if nonzero.any() else 0.0,
        "p90_density_nonzero": float(np.percentile(m[nonzero], 90)) if nonzero.any() else 0.0,
        "p99_density_nonzero": float(np.percentile(m[nonzero], 99)) if nonzero.any() else 0.0,
        "max_density": float(m.max()) if m.size else 0.0,
        "total_objects_est": float(m.sum() * pixel_area_deg2),  # integra densidade
    }
    return stats

# Separar dados

outside = mc_used[mc_used["outside_cluster"] == 1]


density_out = create_density_map(outside)

# Estatísticas

stats_out = healpix_density_stats(density_out)

# Centro das MC
rot = (50, -69, 45)

# Zoom
xsize = 500
reso  = 5.0  # arcmin/pixel

plt.figure(figsize=(14, 6))


hp.gnomview(
    density_out,
    rot=rot,
    xsize=xsize,
    reso=reso,
    title=f'Fora dos Aglomerados\n{len(outside):,} objetos',
    unit='objetos/deg²',
    cmap='plasma',
    min=0,
    sub=(1, 2, 2),
    notext=False,
    cbar=True
)

# ✅ Pega os eixos corretos criados pelo healpy e adiciona grid
fig = plt.gcf()
axes = fig.get_axes()

# Os dois primeiros eixos costumam ser os mapas; as colorbars virão depois.
# Vamos tentar pegar os dois primeiros "Axes" que não são colorbar.
map_axes = [ax for ax in axes if ax.get_label() != '<colorbar>'][:2]

for ax in map_axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- imprime estatísticas
def pretty_print_stats(name, s):
    print(f"\n=== {name} ===")
    print(f"npix_nonzero        : {s['npix_nonzero']:,} / {s['npix_total']:,}")
    print(f"area_covered_deg2   : {s['area_covered_deg2']:.3f}")
    print(f"total_objects_est   : {s['total_objects_est']:.0f}")
    print(f"mean_density        : {s['mean_density_nonzero']:.3f} obj/deg²")
    print(f"median_density      : {s['median_density_nonzero']:.3f} obj/deg²")
    print(f"p90_density         : {s['p90_density_nonzero']:.3f} obj/deg²")
    print(f"p99_density         : {s['p99_density_nonzero']:.3f} obj/deg²")
    print(f"max_density         : {s['max_density']:.3f} obj/deg²")

pretty_print_stats("FORA aglomerados", stats_out)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Configurar a figura
plt.figure(figsize=(10, 6))
mc_fields_inside = mc_fields_clean[(mc_fields_clean['outside_cluster'] == False) & (mc_fields_clean["unique_values_kept"] == 1)& (mc_fields_clean['h_alpha_emitters_8sigma'] == 1)]
# Plotar o scatter dos dados principais
sns.scatterplot(y=mc_fields_inside['mag_psf_j0515'] - mc_fields_inside['mag_psf_j0660'], 
                x=mc_fields_inside['mag_psf_j0660'] - mc_fields_inside['mag_psf_j0861'], 
                label='mc_fields_clean Hα Emitters', color='gray', s=20, alpha=0.6)
# Plotar o scatter das estrelas Be
sns.scatterplot(y=be['mag_psf_j0515'] - be['mag_psf_j0660'], 
                x=be['mag_psf_j0660'] - be['mag_psf_j0861'], 
                label='Estrelas Be - Figueiredo et. al 2025', color='blue', s=60, 
                marker='*', edgecolor='black')

# Adicionar o contorno (densidade) para os dados 'be'
# Opção 1: Contorno de densidade (recomendado)
sns.kdeplot(x=be['mag_psf_j0660'] - be['mag_psf_j0861'],
            y=be['mag_psf_j0515'] - be['mag_psf_j0660'],
            color='blue', linewidths=3, levels=5)

# Opção 2: Se preferir contornos mais suaves
# sns.kdeplot(x=be['mag_psf_j0660'] - be['mag_psf_j0861'],
#             y=be['mag_psf_j0515'] - be['mag_psf_j0660'],
#             fill=True, alpha=0.3, color='blue', thresh=0.1)

# Opção 3: Se quiser um contorno com hexbin (para muitos pontos)
# plt.hexbin(x=be['mag_psf_j0660'] - be['mag_psf_j0861'],
#            y=be['mag_psf_j0515'] - be['mag_psf_j0660'],
#            gridsize=30, cmap='Blues', alpha=0.7)

# Configurações dos eixos
plt.ylim(5, -5)
plt.xlim(3, -5)

plt.axvline(-0.15)
plt.axhline(0.3)

# Rótulos e título
plt.xlabel('F0660 - F0861', fontsize=12)
plt.ylabel('F0515 - F0660', fontsize=12)
plt.title('Diagrama de Cores - Hα Emitters e Estrelas Be (Figueiredo et. al, 2025 & Simbad)', fontsize=14)

# Adicionar grade para melhor visualização
plt.grid(True, alpha=0.3)

# Legenda
plt.legend(loc='best', fontsize=10)

# Ajustar layout
plt.tight_layout()

plt.show()

In [ ]:
be_selection_clusters = mc_fields_clean[(mc_fields_clean['outside_cluster'] == 0) &
                                      (mc_fields_clean["unique_values_kept"] == 1)&
                                        (mc_fields_clean['h_alpha_emitters_8sigma'] == 1) &
                                           (mc_fields_clean['mag_psf_j0515'] - mc_fields_clean['mag_psf_j0660'] < 0.3) &
                                            (mc_fields_clean['mag_psf_j0660'] - mc_fields_clean['mag_psf_j0861'] < -0.15)]

In [ ]:
out

In [ ]:
be_selection_clusters = be_selection_clusters.merge(out[['star_id', 'cluster_Names', 'cluster_ra', 'cluster_dec']], left_on='id', right_on='star_id', how='left')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

mc_used = be_selection_clusters

# Configuração HEALPix
nside = 512
npix = hp.nside2npix(nside)
pixel_area_deg2 = hp.nside2pixarea(nside, degrees=True)

def create_density_map(data, ra_col="ra", dec_col="dec"):
    if len(data) == 0:
        return np.zeros(npix, dtype=float)

    theta = np.deg2rad(90.0 - data[dec_col].to_numpy())  # colatitude
    phi   = np.deg2rad(data[ra_col].to_numpy() % 360.0)  # longitude
    pix = hp.ang2pix(nside, theta, phi)
    counts = np.bincount(pix, minlength=npix).astype(float)
    return counts / pixel_area_deg2

def healpix_density_stats(density_map):
    """
    Estatísticas do mapa de densidade (objetos/deg²).
    Considera apenas pixels com densidade > 0 como 'área coberta'.
    """
    m = np.asarray(density_map, float)
    nonzero = m > 0
    area_deg2 = nonzero.sum() * pixel_area_deg2

    stats = {
        "npix_total": int(m.size),
        "npix_nonzero": int(nonzero.sum()),
        "area_covered_deg2": float(area_deg2),
        "mean_density_nonzero": float(m[nonzero].mean()) if nonzero.any() else 0.0,
        "median_density_nonzero": float(np.median(m[nonzero])) if nonzero.any() else 0.0,
        "p90_density_nonzero": float(np.percentile(m[nonzero], 90)) if nonzero.any() else 0.0,
        "p99_density_nonzero": float(np.percentile(m[nonzero], 99)) if nonzero.any() else 0.0,
        "max_density": float(m.max()) if m.size else 0.0,
        "total_objects_est": float(m.sum() * pixel_area_deg2),  # integra densidade
    }
    return stats

# Separar dados

inside = mc_used[mc_used["outside_cluster"] == 0]


density_inside = create_density_map(inside)

# Estatísticas

stats_inside = healpix_density_stats(density_inside)
# Centro das MC
rot = (50, -69, 45)

# Zoom
xsize = 200
reso  = 10.0  # arcmin/pixel

plt.figure(figsize=(14, 6))


hp.gnomview(
    density_inside,
    rot=rot,
    xsize=xsize,
    reso=reso,
    title=f'Dentro dos Aglomerados\n{len(inside):,} objetos',
    unit='objetos/deg²',
    cmap='viridis',
    min=0,
    sub=(1, 2, 2),
    notext=False,
    cbar=True
)

# ✅ Pega os eixos corretos criados pelo healpy e adiciona grid
fig = plt.gcf()
axes = fig.get_axes()

# Os dois primeiros eixos costumam ser os mapas; as colorbars virão depois.
# Vamos tentar pegar os dois primeiros "Axes" que não são colorbar.
map_axes = [ax for ax in axes if ax.get_label() != '<colorbar>'][:2]

for ax in map_axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- imprime estatísticas
def pretty_print_stats(name, s):
    print(f"\n=== {name} ===")
    print(f"npix_nonzero        : {s['npix_nonzero']:,} / {s['npix_total']:,}")
    print(f"area_covered_deg2   : {s['area_covered_deg2']:.3f}")
    print(f"total_objects_est   : {s['total_objects_est']:.0f}")
    print(f"mean_density        : {s['mean_density_nonzero']:.3f} obj/deg²")
    print(f"median_density      : {s['median_density_nonzero']:.3f} obj/deg²")
    print(f"p90_density         : {s['p90_density_nonzero']:.3f} obj/deg²")
    print(f"p99_density         : {s['p99_density_nonzero']:.3f} obj/deg²")
    print(f"max_density         : {s['max_density']:.3f} obj/deg²")

pretty_print_stats("DENTRO dos aglomerados", stats_inside)

In [ ]:

plt.scatter(mc_fields_clean[(mc_fields_clean["unique_values_kept"] == 1)&(mc_fields_clean['h_alpha_emitters_8sigma'] == 0)]['ra'], mc_fields_clean[(mc_fields_clean["unique_values_kept"] == 1)&(mc_fields_clean['h_alpha_emitters_8sigma'] == 0)]['dec'], color='lightgray', s=0.5, alpha=0.3)
plt.scatter(mc_fields_outside['ra'], mc_fields_outside['dec'], color='red', s=0.5, alpha=0.5)
plt.scatter(mc_fields_inside['ra'], mc_fields_inside['dec'], color='blue', s=0.5, alpha=0.5)


plt.title('H-alpha emitters - spacial distribution over the clouds (inside vs outside clusters)')